# 강의 05 · 실습 3 — 운영 장치 · (3.5) 디버깅


## 1. 문제상황

- 시립도서관의 안내 서비스는 예산이 빠듯해 평소에는 경량 모델을 씁니다. 경량 모델이 응답하지 않을 때만 기본 모델, 그다음 고성능 모델로 넘어가야 합니다.
- 담당자는 하루 예산을 0.0005달러로 잡았고, 예산을 넘기면 그날은 호출을 멈추려 합니다. 0.0005달러는 실습에서 초과를 바로 보기 위해 작게 잡은 값입니다.
- 안내문에는 규정집이 통째로 들어가 매 호출마다 긴 접두가 보내지므로, 캐시가 붙는지 확인해야 합니다.
- 연체·환불·항의처럼 민원성 질문은 답이 틀리면 안 되므로 고성능 모델로 보내고, 그 밖의 질문은 경량 모델로 처리하려 합니다.


## 2. 문제와 목표

- **문제**: 비용 우선 운영에 맞는 대체 순서·예산·캐시 확인·모델 선택 규칙이 없습니다. 도서관의 사정에 맞지 않는 장치를 붙이면 비용이 오히려 늘어납니다.
  - 아래 「6. 코드 — 스텝바이스텝」의 코드는 이 목표를 잘못 구현한 완성 코드이며 결함이 세 개 있습니다. 실행해 증상을 관찰하고 「4. 단계별 요구사항」과 한 줄씩 대조해 셋을 모두 찾아 고칩니다.
- **목표**
  - 서비스 코드를 고치지 않고 호출 단계에 운영 장치 여섯 개를 비용 우선 순서로 붙입니다.
    - 대체(fallback): 1차 경량 → 2차 기본 → 3차 고성능
    - 실패 런 기록, 세 모델 비용 표, 규정집 접두(FAQ 열두 번 반복)의 캐시 히트 확인
    - 토큰 예산: 누적 비용이 0.0005달러 이상이면 멈춤(경량 모델을 반복 호출)
    - 티어링: 질문에 「연체」「환불」「항의」「분실」 가운데 하나라도 있으면 고성능, 아니면 경량 모델
    - 티어링에 넣는 질문 세 개는 「6. 코드 — 스텝바이스텝」 단계 ⑥에 주어져 있습니다
- **목표 달성 여부의 판정 기준**:
  - ① 존재하지 않는 1차 모델로 불러도 2차 기본 모델(`gpt-5.6-luna`)이 답합니다.
  - ② 실패 런이 예외 이름과 함께 출력됩니다.
  - ③ 세 모델의 비용이 표로 출력되고 경량 모델이 가장 쌉니다.
  - ④ 규정집 접두의 2회차 호출에서 캐시 히트가 0보다 큽니다.
  - ⑤ 누적 비용이 0.0005달러를 넘긴 회차에서 `BudgetExceededError`로 멈춥니다.
  - ⑥ 민원성 질문(연체·환불·항의·분실 가운데 하나라도 든 질문)은 고성능 모델, 그 밖의 질문은 경량 모델로 출력됩니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec05_ex03_s3_diagram.svg)


## 4. 단계별 요구사항

1. **대체(fallback)를 붙입니다.**
    - `completion_with_ladder(messages, primary)`는 `litellm.completion`을 `fallbacks=[PRIMARY, HIGH]`와 함께 부릅니다.
    - 1차는 경량 모델 `CHEAP`이며, 시연에서는 1차로 존재하지 않는 이름 `GHOST`를 넣어 2차 기본 모델이 답하는지 보고 답한 모델 이름을 출력합니다.
2. **실패 런을 남깁니다.**
    - 대체 없이 `GHOST`를 부르고 예외를 받아 예외 이름과 메시지 앞부분을 출력한 뒤 `Client().flush()`를 부릅니다.
3. **비용을 잽니다.**
    - 모델 세 개를 같은 질문으로 한 번씩 부르고 입력 토큰·출력 토큰·비용을 표로 출력합니다.
4. **캐시 히트를 확인합니다.**
    - FAQ를 열두 번 반복한 규정집 접두(`LONG_GUIDE`, 맨 앞에 실행 시각 표시)를 시스템 메시지로 넣어 같은 질문을 두 번 부르고, 회차마다 입력 토큰·캐시 히트 토큰·비용을 출력합니다.
5. **예산 장치를 붙입니다.**
    - `guarded_completion(**kwargs)`는 누적 비용이 예산 `BUDGET`(0.0005달러) 이상이면 `litellm.BudgetExceededError`를 내고, 아니면 호출한 뒤 비용을 누적합니다.
    - 경량 모델로 같은 질문을 최대 열다섯 번 반복해 몇 회차에서 멈추는지 출력합니다.
6. **티어링을 붙입니다.**
    - `choose_model(question)`은 질문에 「연체」「환불」「항의」「분실」 가운데 하나라도 있으면 `HIGH`, 아니면 `CHEAP`를 돌려줍니다.
    - 질문 세 개를 고른 모델로 부르고 모델 이름·답 앞부분·비용을 출력한 뒤 `Client().flush()`를 부릅니다.
    - 출력 형식은 단계 ②가 「실패 런: 예외 이름 - 메시지 앞부분」, 단계 ⑤가 「N회차 OK · 누적 X / 예산 Y」와 「N회차 멈춤: BudgetExceededError · 누적 X / 예산 Y」입니다.
    - 아래 「6. 코드 — 스텝바이스텝」의 코드는 위 여섯 개의 요구사항을 잘못 구현한 것입니다.
    - 요구사항 한 줄과 코드 한 줄을 짝지어 읽으면서 어긋나는 곳을 찾습니다.
    - 결함은 문법 오류가 아니라 「요구사항이 시킨 것과 코드가 하는 일이 다른 곳」입니다.


## 5. 코드 골격 — 운영 장치 6단

이미 도는 서비스의 호출 단계에 운영 장치를 붙이는 순서는 다음 여섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 여섯 단계와 하나씩 대응합니다. 여섯 단계 어디에서도 서비스 코드를 고치지 않습니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 대체(fallback) | 1차 모델이 실패하면 2차·3차로 넘어갑니다 | `litellm.completion(..., fallbacks=[PRIMARY, HIGH])` | 1 |
| ② 실패 런 기록 | 대체 없는 실패를 예외로 받고 실패 런으로 남깁니다 | `try`/`except`, `Client().flush()` | 2 |
| ③ 비용 측정 | 모델마다 토큰과 비용을 잽니다 | `litellm.completion_cost(res)`, `res.usage` | 3 |
| ④ 프롬프트 캐싱 | 같은 긴 접두를 두 번 보내 캐시 히트를 확인합니다 | `usage.prompt_tokens_details.cached_tokens` | 4 |
| ⑤ 토큰 예산 | 누적 비용이 예산을 넘으면 호출을 멈춥니다 | `litellm.BudgetExceededError` | 5 |
| ⑥ 티어링 | 질문에 따라 경량·고성능 모델을 고릅니다 | `choose_model(question)` | 6 |


## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고, 키를 읽고, 추적 설정을 켜고, 모델 이름 세 개를 정합니다. 이 실습의 모델 호출은 `litellm.completion`을 직접 씁니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- 이 실습은 OpenAI 키와 LangSmith 키를 함께 씁니다. `.env` 파일에는 다음 두 줄을 넣습니다.
- 모델 세 개는 기본 모델(`PRIMARY`), 경량 모델(`CHEAP`), 고성능 모델(`HIGH`)입니다. 모델 이름은 공급자 이름을 앞에 붙인 문자열 그대로 쓰고, 별칭이나 중계 서버는 쓰지 않습니다.
- `litellm.suppress_debug_info = True`와 `logging` 설정 한 줄은 오류가 났을 때 litellm이 화면에 출력하는 안내 배너와 오류 로그를 끕니다. 동작에는 영향이 없습니다. 실패 자체는 단계 ②에서 예외로 받아 직접 출력합니다.

```
OPENAI_API_KEY=발급받은_키
LANGSMITH_API_KEY=발급받은_키
```


In [ ]:
import logging
import os
import time
import warnings

from dotenv import load_dotenv, find_dotenv

import litellm
from langsmith import Client, traceable
from langsmith.run_helpers import get_current_run_tree

warnings.filterwarnings("ignore", message="Pydantic serializer warnings")   # 추적 라이브러리가 내는 직렬화 경고를 화면에서 감춥니다
litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
if not os.environ.get("LANGSMITH_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 LANGSMITH_API_KEY 한 줄을 넣습니다.")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "sesac-lec05-ex03"

PRIMARY = "openai/gpt-5.6-luna"
CHEAP = "openai/gpt-4o-mini"
HIGH = "openai/gpt-5.6-terra"
print("모델 세 개:", PRIMARY, CHEAP, HIGH)

운영 장치를 붙일 도서관 안내 서비스입니다. 이미 만들어 둔 부품이며, 이 실습에서는 고치지 않습니다. `make_messages`가 질문을 받아 FAQ를 근거로 넣은 메시지 목록을 만들고, 호출 단계 `litellm.completion`에는 llm 런 기록 계측이 붙어 있습니다. 도서관은 평소 경량 모델을 씁니다.


In [ ]:
FAQ = """
[대출] Q: 책은 몇 권까지 며칠 동안 빌릴 수 있나요?
A: 회원 한 사람이 5권까지 14일 동안 빌릴 수 있습니다.
[연장] Q: 대출 기간을 연장할 수 있나요?
A: 반납 예정일 전에 홈페이지에서 1회 7일 연장할 수 있습니다. 예약자가 있는 책은 연장할 수 없습니다.
[휴관] Q: 휴관일은 언제인가요?
A: 매주 월요일과 법정 공휴일에 휴관합니다.
[연체] Q: 반납이 늦으면 어떻게 되나요?
A: 연체한 날수만큼 대출이 정지됩니다. 연체료는 없습니다.
"""

GUIDE = ("너는 시립도서관의 안내 담당자다. 아래 FAQ에 적힌 내용만 근거로 두 문장 안에서 답한다. "
         "인사말에는 짧은 인사로 답한다. FAQ에 없는 내용을 물으면 '해당 내용은 확인할 수 없습니다.'라고만 답한다.")


def make_messages(question: str, guide: str = GUIDE) -> list:
    """질문을 받아 FAQ를 근거로 넣은 메시지 목록을 만든다."""
    return [{"role": "system", "content": guide + "\n=== FAQ ===\n" + FAQ},
            {"role": "user", "content": question}]


_completion = litellm.completion


@traceable(run_type="llm", name="litellm.completion", metadata={"ls_provider": "openai"})
def completion(**kwargs):
    """이 실습에서 쓰는 관측 계측. 답한 모델 이름을 런 메타데이터에 적는다."""
    res = _completion(**kwargs)
    get_current_run_tree().metadata["ls_model_name"] = res.model
    return res


litellm.completion = completion

Q = "책은 몇 권까지 며칠 동안 빌릴 수 있나요?"
res = litellm.completion(model=CHEAP, messages=make_messages(Q))
print(res.model, "→", res.choices[0].message.content[:60])

### 단계 ① — 대체(fallback) (요구사항 1)

비용 우선 운영에서는 싼 모델이 1차입니다. `fallbacks`의 순서가 곧 대체 순서이므로 2차 기본 모델, 3차 고성능 모델 순으로 적습니다.


In [ ]:
GHOST = "openai/gpt-4o-mini-nonexistent"   # 1차 경량 모델 장애를 모의로 만드는 존재하지 않는 이름
LADDER = [HIGH, PRIMARY]                     # 2차 → 3차


def completion_with_ladder(messages: list, primary: str = CHEAP):
    """1차 경량 모델이 실패하면 대체 순서를 따라 기본·고성능 모델로 넘어간다."""
    return litellm.completion(model=primary, messages=messages, fallbacks=LADDER)


res = completion_with_ladder(make_messages(Q), primary=GHOST)
print("1차:", GHOST)
print("답한 모델:", res.model)
print(res.choices[0].message.content[:60])

### 단계 ② — 실패 런 기록 (요구사항 2)

대체가 없으면 실패는 예외로 올라옵니다. 예외를 받아 화면에 출력하되, 호출 단계에 붙은 관측 계측이 같은 호출을 LangSmith에 실패 런으로 남깁니다. 실패 런에는 예외 메시지가 그대로 붙어, 나중에 무엇이 왜 실패했는지 화면에서 찾을 수 있습니다.


In [ ]:
try:
    litellm.completion(model=GHOST, messages=make_messages(Q))
    print("예상 밖 성공")
except Exception as e:
    print("실패 런:", type(e).__name__, "-", str(e)[:80])

Client().flush()
print("실패 런이 LangSmith 프로젝트 sesac-lec05-ex03에 남았습니다.")

### 단계 ③ — 비용 측정 (요구사항 3)

`litellm.completion_cost(res)`는 응답의 토큰 수와 모델의 가격표로 비용을 계산합니다. 같은 질문이라도 모델마다 비용이 다르므로, 표로 나란히 놓고 봅니다.


In [ ]:
print(f"{'모델':<22} {'입력 토큰':>6} {'출력 토큰':>6} {'비용(달러)':>12}")
for model in (CHEAP, PRIMARY, HIGH):
    res = litellm.completion(model=model, messages=make_messages(Q))
    cost = litellm.completion_cost(res)
    print(f"{model:<22} {res.usage.prompt_tokens:>8} {res.usage.completion_tokens:>8} {cost:>12.6f}")

### 단계 ④ — 프롬프트 캐싱 (요구사항 4)

같은 긴 접두(시스템 메시지)를 두 번 연속 보내면 2회차에는 접두의 대부분이 캐시에서 읽혀 입력 비용이 줄어듭니다. 캐시가 붙으려면 접두가 충분히 길어야 하므로 FAQ를 열두 번 반복해 넣습니다. 캐시는 몇 분 동안 유지되므로, 접두 맨 앞에 실행 시각 표시를 붙여 실행마다 1회차는 캐시 없이 시작하게 합니다. 캐시 히트 토큰은 응답 `usage`의 `prompt_tokens_details.cached_tokens`에 적혀 있습니다.


In [ ]:
LONG_GUIDE = GUIDE + "\n=== FAQ ===\n" + (FAQ + "\n") * 12

for i in (1, 2):
    tag = f"[실행 {time.strftime('%H%M%S')} {time.time()}] "
    res = litellm.completion(model=PRIMARY, messages=[{"role": "system", "content": tag + LONG_GUIDE},
                                                      {"role": "user", "content": Q}])
    usage = res.usage.model_dump()
    cached = (usage.get("prompt_tokens_details") or {}).get("cached_tokens") or 0
    print(f"{i}회차: 입력 토큰 {usage['prompt_tokens']} · 캐시 히트 {cached} · 비용 {litellm.completion_cost(res):.6f}")

### 단계 ⑤ — 토큰 예산 (요구사항 5)

예산 장치는 호출 단계를 한 번 더 감싸 누적 비용을 셉니다. 누적이 예산 이상이면 호출하지 않고 `litellm.BudgetExceededError`를 냅니다. litellm에는 `max_budget` 설정이 있지만 이 환경의 litellm 판(1.97.0)은 응답 객체의 비용을 누적하지 않아 발동하지 않으므로, 누적은 직접 하고 예외 클래스만 litellm의 것을 씁니다.


In [ ]:
BUDGET = 0.0005   # 달러 (하루 예산)
spent = 0.0


def guarded_completion(**kwargs):
    """누적 비용이 예산 이상이면 호출하지 않고 멈춘다. 호출했으면 비용을 누적한다."""
    global spent
    if spent >= BUDGET:
        raise litellm.BudgetExceededError(current_cost=spent, max_budget=BUDGET)
    res = litellm.completion(**kwargs)
    spent = litellm.completion_cost(res)
    return res


for i in range(1, 16):
    try:
        guarded_completion(model=CHEAP, messages=make_messages(Q))
        print(f"{i}회차 OK · 누적 {spent:.6f} / 예산 {BUDGET}")
    except litellm.BudgetExceededError:
        print(f"{i}회차 멈춤: BudgetExceededError · 누적 {spent:.6f} / 예산 {BUDGET}")
        break

### 단계 ⑥ — 티어링 (요구사항 6)

도서관의 티어링 규칙은 질문의 길이가 아니라 민원성 낱말입니다. 답이 틀리면 안 되는 질문만 고성능 모델로 보냅니다.


In [ ]:
COMPLAINT_WORDS = ("연체", "환불", "항의", "분실")


def choose_model(question: str) -> str:
    """민원성 낱말이 있으면 고성능 모델, 없으면 경량 모델."""
    if any(word in question for word in COMPLAINT_WORDS):
        return HIGH
    return CHEAP


QUESTIONS = [
    "휴관일은 언제인가요?",
    "대출 기간을 연장할 수 있나요?",
    "빌린 책을 분실했는데 연체 처리까지 되면 어떻게 항의하나요?",
]
for question in QUESTIONS:
    model = choose_model(question)
    res = litellm.completion(model=model, messages=make_messages(question))
    print(f"[{model.split('/')[-1]:<13}] {question[:22]} → {res.choices[0].message.content[:36]} ({litellm.completion_cost(res):.6f}달러)")

Client().flush()

## 7. 실행 결과 확인

결함을 고친 뒤 다시 실행해 다음 세 가지를 확인합니다.

1. 단계 ①에서 답한 모델이 `gpt-5.6-luna`입니다. 고성능 모델이 아닙니다.
2. 단계 ④에서 2회차의 캐시 히트가 0보다 큽니다.
3. 단계 ⑤에서 누적 비용이 회차마다 늘어나고, 예산 0.0005달러를 넘긴 회차에서 「멈춤: BudgetExceededError」가 출력됩니다.

고치기 전에는 답한 모델이 고성능 모델이고, 캐시 히트가 두 번 다 0이며, 열다섯 번을 다 불러도 멈추지 않습니다. 세 증상이 각각 결함 하나씩을 가리킵니다.
